In [ ]:
# Phase 2: 时间标准化与特征工程

**目的**: 统一时间戳格式，提取5G语义特征，为图构建做准备

**输入**: Phase 1分析结果和原始数据

**输出**: 标准化时间戳和5G语义特征


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from datetime import datetime, timedelta

print("Phase 2: 时间标准化与特征工程环境准备完成")


Phase 2: 时间标准化与特征工程环境准备完成


In [ ]:
## 2.1 时间戳标准化函数


In [2]:
def standardize_timestamps(df, base_timestamp=1663200000):
    """将MM:SS.s格式转换为Unix时间戳"""
    
    def parse_time_string(time_str):
        if pd.isna(time_str) or time_str == '' or time_str == 'nan':
            return np.nan
        
        try:
            # 解析MM:SS.s格式
            time_str = str(time_str)
            if ':' in time_str:
                parts = time_str.split(':')
                if len(parts) == 2:
                    minutes = int(parts[0])
                    seconds_parts = parts[1].split('.')
                    seconds = int(seconds_parts[0])
                    microseconds = int(seconds_parts[1]) if len(seconds_parts) > 1 else 0
                    
                    total_seconds = minutes * 60 + seconds + microseconds / 10
                    return base_timestamp + total_seconds
            
            # 如果不包含冒号，尝试直接转换为浮点数
            return base_timestamp + float(time_str)
        except (ValueError, TypeError):
            return np.nan
    
    # 创建副本并使用np.where避免NumPy兼容性问题
    df_copy = df.copy()
    
    # 安全的时间字段类型转换
    start_series = df_copy['StartTime']
    df_copy['StartTime'] = np.where(pd.isna(start_series), '', start_series.astype(str))
    
    last_series = df_copy['LastTime']
    df_copy['LastTime'] = np.where(pd.isna(last_series), '', last_series.astype(str))
    
    # 转换时间戳
    df_copy['StartTime_unix'] = df_copy['StartTime'].apply(parse_time_string)
    df_copy['LastTime_unix'] = df_copy['LastTime'].apply(parse_time_string)
    df_copy['Duration'] = df_copy['LastTime_unix'] - df_copy['StartTime_unix']
    
    return df_copy

print("时间戳标准化函数定义完成")


时间戳标准化函数定义完成


In [3]:
# 直接从原始CSV加载数据（避免pickle版本兼容性问题）
print("从原始CSV加载数据...")
df = pd.read_csv('../data/raw/BTS1_BTS2_fields_preserved.csv')
print(f"加载数据完成: {len(df)} 条记录")

# 执行基本数据清洗（来自Phase 1的逻辑）
print("\\n执行基本数据清洗...")
initial_count = len(df)

# 去除重复记录
df = df.drop_duplicates()
print(f"去重后: {len(df)} 条记录 (删除 {initial_count - len(df)} 重复记录)")

# 处理缺失值 - 仅删除关键字段全为空的记录
key_fields = ['StartTime', 'Proto', 'SrcAddr', 'DstAddr']
df = df.dropna(subset=key_fields, how='all')
print(f"清洗后: {len(df)} 条记录")

def extract_5g_semantic_features(df):
    """提取5G语义特征"""
    
    # 协议类型映射
    protocol_mapping = {
        'tcp': 'TCP',
        'udp': 'UDP', 
        'icmp': 'ICMP',
        'sctp': 'SCTP',  # 5G信令协议
        'arp': 'ARP'
    }
    
    # 使用更安全的数据类型转换方法
    df_copy = df.copy()
    
    # 使用where方法代替fillna，避免NumPy兼容性问题
    proto_series = df_copy['Proto']
    df_copy['Proto'] = np.where(pd.isna(proto_series), 'unknown', proto_series.astype(str))
    
    # 标准化协议名称
    df_copy['Proto_normalized'] = df_copy['Proto'].str.lower().map(protocol_mapping).fillna(df_copy['Proto'])
    
    # 5G控制面识别 (SCTP协议通常用于5G信令)
    df_copy['is_5g_control_plane'] = df_copy['Proto_normalized'] == 'SCTP'
    
    # QoS标记解析 (从sTos和dTos字段) - 使用np.where避免兼容性问题
    sTos_series = df_copy['sTos']
    df_copy['qos_class_src'] = np.where(pd.isna(sTos_series), 'unknown', sTos_series.astype(str))
    
    dTos_series = df_copy['dTos']
    df_copy['qos_class_dst'] = np.where(pd.isna(dTos_series), 'unknown', dTos_series.astype(str))
    
    # 流量方向性特征
    df_copy['traffic_ratio'] = pd.to_numeric(df_copy['SrcBytes'], errors='coerce') / (pd.to_numeric(df_copy['DstBytes'], errors='coerce') + 1)
    df_copy['packet_ratio'] = pd.to_numeric(df_copy['SrcPkts'], errors='coerce') / (pd.to_numeric(df_copy['DstPkts'], errors='coerce') + 1)
    
    # 5G特征标记
    df_copy['has_5g_features'] = (
        df_copy['is_5g_control_plane'] | 
        df_copy['qos_class_src'].str.contains('ef|af|cs', case=False, na=False) |
        df_copy['qos_class_dst'].str.contains('ef|af|cs', case=False, na=False)
    )
    
    return df_copy

# 执行5G语义特征提取
print("\\n提取5G语义特征...")
df = extract_5g_semantic_features(df)
print("5G语义特征提取完成")

# 统计5G特征
print(f"\\n5G控制面流量: {df['is_5g_control_plane'].sum()} 条记录")
print(f"包含5G特征的流量: {df['has_5g_features'].sum()} 条记录")
print(f"SCTP协议记录: {(df['Proto_normalized'] == 'SCTP').sum()} 条")


从Phase 1加载清洗后的数据...


ModuleNotFoundError: No module named 'numpy._core'

In [ ]:
# 执行时间标准化
print("\\n执行时间标准化...")
df_processed = standardize_timestamps(df)

# 验证时间转换效果
conversion_success = df_processed['StartTime_unix'].notna().sum()
total_records = len(df_processed)
conversion_rate = conversion_success / total_records

print(f"时间戳转换成功率: {conversion_rate:.2%}")
print(f"成功转换: {conversion_success:,} / {total_records:,} 条记录")

# 显示时间转换示例
print("\\n时间转换示例:")
sample_indices = df_processed['StartTime_unix'].notna()
sample_df = df_processed[sample_indices].head(5)
for i, row in sample_df.iterrows():
    print(f"原始: {row['StartTime']} -> Unix: {row['StartTime_unix']:.2f} -> 持续时间: {row['Duration']:.2f}s")

# 时间分布可视化
plt.figure(figsize=(15, 10))

# 1. 时间戳分布
plt.subplot(2, 3, 1)
valid_start_times = df_processed['StartTime_unix'].dropna()
plt.hist(valid_start_times, bins=50, alpha=0.7)
plt.title('StartTime Unix时间戳分布')
plt.xlabel('Unix Timestamp')

# 2. 持续时间分布
plt.subplot(2, 3, 2)
valid_durations = df_processed['Duration'].dropna()
valid_durations = valid_durations[valid_durations >= 0]  # 过滤负值
plt.hist(valid_durations, bins=50, alpha=0.7)
plt.title('Flow Duration Distribution')
plt.xlabel('Duration (seconds)')

# 3. 5G特征分布
plt.subplot(2, 3, 3)
feature_counts = df_processed['has_5g_features'].value_counts()
plt.pie(feature_counts.values, labels=['无5G特征', '有5G特征'], autopct='%1.1f%%')
plt.title('5G特征分布')

# 4. 协议标准化效果
plt.subplot(2, 3, 4)
proto_counts = df_processed['Proto_normalized'].value_counts()
plt.bar(proto_counts.index, proto_counts.values)
plt.title('标准化Protocol Distribution')
plt.xticks(rotation=45)

# 5. QoS Class Distribution
plt.subplot(2, 3, 5)
qos_counts = df_processed['qos_class_src'].value_counts().head(10)
plt.bar(range(len(qos_counts)), qos_counts.values)
plt.title('QoS Class Distribution (Top 10)')
plt.xticks(range(len(qos_counts)), qos_counts.index, rotation=45)

# 6. 流量比例分布
plt.subplot(2, 3, 6)
traffic_ratios = df_processed['traffic_ratio'].dropna()
traffic_ratios = traffic_ratios[traffic_ratios < 100]  # 过滤极值
plt.hist(traffic_ratios, bins=50, alpha=0.7)
plt.title('流量比例分布')
plt.xlabel('Traffic Ratio')

plt.tight_layout()
plt.savefig('../data/processed/metadata/time_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 保存处理后的数据（使用CSV避免版本兼容性问题）
print("\\n保存时间标准化后的数据...")
df_processed.to_csv('../data/processed/time_standardized.csv', index=False)
print(f"时间标准化数据已保存: {len(df_processed)} 条记录")

# 生成质量报告
time_quality_report = {
    "timestamp": datetime.now().isoformat(),
    "total_records": len(df_processed),
    "time_conversion_rate": float(conversion_rate),
    "successful_conversions": int(conversion_success),
    "5g_control_plane_flows": int(df_processed['is_5g_control_plane'].sum()),
    "flows_with_5g_features": int(df_processed['has_5g_features'].sum()),
    "protocol_distribution": df_processed['Proto_normalized'].value_counts().to_dict(),
    "average_duration": float(valid_durations.mean()) if len(valid_durations) > 0 else 0,
    "duration_std": float(valid_durations.std()) if len(valid_durations) > 0 else 0
}

with open('../data/processed/metadata/time_quality_report.json', 'w', encoding='utf-8') as f:
    json.dump(time_quality_report, f, indent=2, ensure_ascii=False)

print("\\n✓ Phase 2 完成:")
print(f"  - 时间标准化数据: ../data/processed/time_standardized.csv")
print(f"  - 质量报告: ../data/processed/metadata/time_quality_report.json")
print(f"  - 时间分布图表: ../data/processed/metadata/time_distribution.png")
